In [1]:
import pandas as pd
train_df = pd.read_csv(r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\train.csv")
valid_df = pd.read_csv(r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\val.csv")
test_df = pd.read_csv(r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\test.csv")

In [2]:
train_df.sort_values(by="timestamp", inplace=True)
valid_df.sort_values(by="timestamp", inplace=True)
test_df.sort_values(by="timestamp", inplace=True)
cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [3]:
#Base learner: train trên tập train, early stopping trên tập valid. Meta learner: train trên tập valid (tập meta validation), đánh giá trên tập test
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Kiểm tra GPU
print(f"TensorFlow phiên bản: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"Số lượng GPU khả dụng cho TensorFlow: {len(gpus)}")
if gpus:
    for gpu in gpus:
        print("Trạng thái GPU:", gpu)

# ---------------------------------------------------------
# 1. TÁCH DỮ LIỆU & ĐỊNH DẠNG ĐẦU VÀO
# ---------------------------------------------------------
print("Chuẩn bị dữ liệu...")
# Lấy X, y từ các tập dữ liệu gốc
X_train = train_df.drop(columns=['label']).values
y_train = train_df['label'].values

X_valid = valid_df.drop(columns=['label']).values
y_valid = valid_df['label'].values

X_test = test_df.drop(columns=['label']).values
y_test = test_df['label'].values

num_classes = len(np.unique(y_train))
print(f"Số lượng classes: {num_classes}")

# Reshape X sang dạng 3D (samples, time_steps=1, features) cho LSTM/GRU
X_train_3d = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_valid_3d = X_valid.reshape(X_valid.shape[0], 1, X_valid.shape[1])
X_test_3d = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

# ---------------------------------------------------------
# 2. XÂY DỰNG VÀ HUẤN LUYỆN CÁC MÔ HÌNH CƠ SỞ (TẦNG 1)
# ---------------------------------------------------------
print("\n--- Bắt đầu huấn luyện Tầng 1 trên TẬP TRAIN, Early Stopping trên TẬP VALID ---")

# 1. Gradient Boosting (Scikit-learn không hỗ trợ eval_set bên ngoài cho Early Stopping)
print("- Đang huấn luyện GBM (HistGradientBoosting trên CPU)...")
gbm = HistGradientBoostingClassifier()
gbm.fit(X_train, y_train)

# 2. LightGBM (Cập nhật Early Stopping)
print("- Đang huấn luyện LightGBM (trên CPU đa luồng)...")
lgbm = LGBMClassifier(verbose=-1, n_jobs=-1)
lgbm.fit(
    X_train, y_train, 
    eval_set=[(X_valid, y_valid)], 
    callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False)]
)

# 3. Bagging (Giữ nguyên, bản chất là ensemble độc lập, không có Early Stopping)

print("- Đang huấn luyện Bagging (100 cây trên 1 luồng CPU để tránh tràn RAM)...")
dt_base = DecisionTreeClassifier(max_depth=15)
bagging = BaggingClassifier(estimator=dt_base, n_estimators=100, max_samples=0.01)
bagging.fit(X_train, y_train)
# 4. XGBoost (Cập nhật Early Stopping vào constructor)
print("- Đang huấn luyện XGBoost (trên GPU)...")
xgb = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='mlogloss', 
    tree_method='hist', 
    device='cuda',
    early_stopping_rounds=10
)
xgb.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=0)

# 5. CatBoost (Cập nhật Early Stopping)
print("- Đang huấn luyện CatBoost (trên GPU)...")
cat = CatBoostClassifier(
    verbose=0, 
    task_type='GPU',
    early_stopping_rounds=10
)
cat.fit(X_train, y_train, eval_set=(X_valid, y_valid))

# ---------------------------------------------------------
# Khối Deep Learning (LSTM & GRU)
# Không còn tách 10% từ X_train nữa, sử dụng thẳng X_valid_3d
# ---------------------------------------------------------
nn_es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train LSTM
print("- Đang huấn luyện LSTM (trên GPU)...")
lstm_model = Sequential([
    LSTM(128, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
lstm_model.fit(
    X_train_3d, y_train, 
    validation_data=(X_valid_3d, y_valid), 
    epochs=30, batch_size=1024, 
    callbacks=[nn_es], verbose=1
)

# Train GRU
print("- Đang huấn luyện GRU (trên GPU)...")
gru_model = Sequential([
    GRU(128, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

gru_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
gru_model.fit(
    X_train_3d, y_train, 
    validation_data=(X_valid_3d, y_valid), 
    epochs=20, batch_size=1024, 
    callbacks=[nn_es], verbose=1
)

TensorFlow phiên bản: 2.19.0
Số lượng GPU khả dụng cho TensorFlow: 0
Chuẩn bị dữ liệu...
Số lượng classes: 8

--- Bắt đầu huấn luyện Tầng 1 trên TẬP TRAIN, Early Stopping trên TẬP VALID ---
- Đang huấn luyện GBM (HistGradientBoosting trên CPU)...
- Đang huấn luyện LightGBM (trên CPU đa luồng)...
- Đang huấn luyện Bagging (100 cây trên 1 luồng CPU để tránh tràn RAM)...
- Đang huấn luyện XGBoost (trên GPU)...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\callback.py:385: UserWarning: [10:13:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


- Đang huấn luyện CatBoost (trên GPU)...
- Đang huấn luyện LSTM (trên GPU)...
Epoch 1/30


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.8812 - val_loss: 0.4868
Epoch 2/30
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4641 - val_loss: 0.4325
Epoch 3/30
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4038 - val_loss: 0.4257
Epoch 4/30
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3763 - val_loss: 0.4012
Epoch 5/30
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3707 - val_loss: 0.4052
Epoch 6/30
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3567 - val_loss: 0.4020
Epoch 7/30
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3557 - val_loss: 0.4079
- Đang huấn luyện GRU (trên GPU)...
Epoch 1/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.9282 - val_loss: 0.4957
Epoch 2/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4837 - val_loss: 0.4503
Epoch 3/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4152 - val_loss: 0.4409
Epoch 4/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3912 - val_loss: 0.4092
Epoch 5/20
156/156 ━━━━━━━━━━

In [4]:
print("\n--- Trích xuất Meta-Features từ Tầng 1 ---")
import gc

def get_meta_features(X_2d, X_3d):
    print("  -> Predicting GBM...")
    p_gbm = gbm.predict_proba(X_2d)
    
    print("  -> Predicting LGBM...")
    p_lgbm = lgbm.predict_proba(X_2d)
    
    print("  -> Predicting Bagging...")
    # Nếu đoạn này quá chậm, bạn cần giảm n_estimators của Bagging xuống 100 ở ô huấn luyện trước đó
    p_bag = bagging.predict_proba(X_2d)
    
    print("  -> Predicting XGB...")
    p_xgb = xgb.predict_proba(X_2d)
    
    print("  -> Predicting CatBoost...")
    p_cat = cat.predict_proba(X_2d)
    
    print("  -> Predicting LSTM...")
    # Sửa quan trọng: THÊM BATCH_SIZE và BẬT VERBOSE để tránh kẹt RAM và thấy được thanh tiến trình
    p_lstm = lstm_model.predict(X_3d, batch_size=2048, verbose=1)
    
    print("  -> Predicting GRU...")
    p_gru = gru_model.predict(X_3d, batch_size=2048, verbose=1)
    
    print("  -> Đang nối các đặc trưng (Concatenating)...")
    res = np.hstack([p_gbm, p_lgbm, p_bag, p_xgb, p_cat, p_lstm, p_gru])
    
    # Giải phóng RAM lập tức
    del p_gbm, p_lgbm, p_bag, p_xgb, p_cat, p_lstm, p_gru
    gc.collect()
    
    return res

print("- Tạo dữ liệu huấn luyện Meta-Model trên tập VALID (Hold-out)...")
X_meta_train = get_meta_features(X_valid, X_valid_3d)

print("- Tạo dữ liệu đánh giá Meta-Model trên tập TEST...")
X_meta_test = get_meta_features(X_test, X_test_3d)


# ---------------------------------------------------------
# 4. HUẤN LUYỆN META-MODEL (TẦNG 2) VÀ ĐÁNH GIÁ TRÊN TEST
# ---------------------------------------------------------
print("\n--- Huấn luyện Meta-Model (EnsembleGuard Neural Network) Tầng 2 ---")
meta_model_nn = Sequential([
    Dense(64, activation='relu', input_shape=(X_meta_train.shape[1],)),
    Dense(num_classes, activation='softmax')
])

meta_model_nn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Vẫn giữ nguyên việc huấn luyện trên tập meta validation (X_meta_train, y_valid)
meta_model_nn.fit(X_meta_train, y_valid, epochs=20, batch_size=1024, verbose=1)


print("\n==============================================")
print("ĐÁNH GIÁ ENSEMBLE TRÊN TẬP TEST ĐỘC LẬP TỪ META-MODEL")
print("==============================================")

meta_preds_proba = meta_model_nn.predict(X_meta_test)
meta_preds_classes = np.argmax(meta_preds_proba, axis=1)

print("\n--- Báo cáo Phân loại (Classification Report) ---")
print(classification_report(y_test, meta_preds_classes, digits=4))

acc = accuracy_score(y_test, meta_preds_classes)
prec = precision_score(y_test, meta_preds_classes, average='macro', zero_division=0)
rec = recall_score(y_test, meta_preds_classes, average='macro', zero_division=0)
f1 = f1_score(y_test, meta_preds_classes, average='macro', zero_division=0)

print(f"\n=> Tỷ lệ chính xác (Accuracy): {acc:.4f}")
print(f"=> Macro Precision:          {prec:.4f}")
print(f"=> Macro Recall:             {rec:.4f}")
print(f"=> Macro F1-Score:           {f1:.4f}")


--- Trích xuất Meta-Features từ Tầng 1 ---
- Tạo dữ liệu huấn luyện Meta-Model trên tập VALID (Hold-out)...
  -> Predicting GBM...
  -> Predicting LGBM...
  -> Predicting Bagging...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  -> Predicting XGB...
  -> Predicting CatBoost...
  -> Predicting LSTM...
 1/17 ━━━━━━━━━━━━━━━━━━━━ 1s 79ms/step

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [10:13:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
  -> Predicting GRU...
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
  -> Đang nối các đặc trưng (Concatenating)...
- Tạo dữ liệu đánh giá Meta-Model trên tập TEST...
  -> Predicting GBM...
  -> Predicting LGBM...
  -> Predicting Bagging...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  -> Predicting XGB...
  -> Predicting CatBoost...
  -> Predicting LSTM...
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
  -> Predicting GRU...
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
  -> Đang nối các đặc trưng (Concatenating)...

--- Huấn luyện Meta-Model (EnsembleGuard Neural Network) Tầng 2 ---
Epoch 1/20


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4540 - loss: 1.7329  
Epoch 2/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9259 - loss: 0.7634 
Epoch 3/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9373 - loss: 0.3957 
Epoch 4/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9479 - loss: 0.2947 
Epoch 5/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9480 - loss: 0.2602 
Epoch 6/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9494 - loss: 0.2398 
Epoch 7/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9503 - loss: 0.2293 
Epoch 8/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9515 - loss: 0.2211 
Epoch 9/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9523 - loss: 0.2143 
Epoch 10/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9563 - loss: 0.1974 
Epoch 11/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9553 - loss: 0.1982 
Epoch 12/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9556 - l